### RAG Data Ingestion Pipeline

In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [14]:
### Read all the pdf files from the directory and convert into Document Structure using Document Loaders.

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Every page in a PDF is created as a separate document
            # Add source information to Meta Data
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents loaded so far: {len(all_documents)}")
    return all_documents    

all_pdf_documents = process_all_pdfs("../data/pdf")

Multiple definitions in dictionary at byte 0x3a550 for key /Creator
Multiple definitions in dictionary at byte 0x3a569 for key /Producer


Found 2 PDF Files to process
Processing Network Programming.pdf
Loaded 47 pages
Processing MYSQL practice 1.pdf
Loaded 12 pages
Total documents loaded so far: 59


In [20]:
### Chunking using Text Splitters

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [21]:
chunks=split_documents(all_pdf_documents)

Split 59 documents into 164 chunks

Example chunk:
Content: Beej’s Guide to Network Programming
Using Internet Sockets
Brian "Beej" Hall
beej@piratehaven.org
Copyright © 1995-2001 by Brian "Beej" Hall
Revision History
Revision Version 1.0.0 August, 1995 Revise...
Metadata: {'producer': 'pdfTeX13.d', 'creator': 'LaTeX with hyperref package', 'creationdate': 'D:20010503022300', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'source': '../data/pdf/Network Programming.pdf', 'total_pages': 47, 'page': 0, 'page_label': '1', 'source_file': 'Network Programming.pdf', 'file_type': 'pdf'}


In [ ]:
### Embedding using 